# Try Tavily Search with one restaurant

This is a separate experiment. The working OpenAI notebook and `menus.json` stay unchanged. First check whether Tavily finds a current menu and readable source content. This notebook does not call OpenAI or save meals to the app.

Add `TAVILY_API_KEY=your_key_here` to the project’s local `.env`. Keep the key out of notebook cells and outputs. We reuse `httpx` and `python-dotenv`; no new package is needed.

References: [Search API](https://docs.tavily.com/documentation/api-reference/endpoint/search), [credits and pricing](https://docs.tavily.com/documentation/api-credits). A basic search consumes one Tavily credit; free-plan allowances are account-limited, not unlimited search.

## 1. Load the Tavily key

This only loads settings. It makes no network request. Run it again after adding the key to `.env`.

In [ ]:
import os
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import httpx
from dotenv import load_dotenv

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

load_dotenv(project_root / ".env", override=True)
tavily_key = os.getenv("TAVILY_API_KEY")
print("Tavily key loaded:", bool(tavily_key))


## 2. Choose a restaurant

Start with Spilka. We include the requested date in the query but do not filter by page publication date: a weekly menu may have been published before today. Search results alone do not prove the menu is current.

In [ ]:
restaurant = {
    "name": "Spilka",
    "address": "Legionárska 8, Bratislava",
    "website": "https://www.spilkarestaurant.sk/",
    "domain": "spilkarestaurant.sk",
}

today = datetime.now(ZoneInfo("Europe/Bratislava")).date()
query = (
    f"{restaurant['name']} {restaurant['address']} "
    f"denné obedové menu {today.strftime('%d.%m.%Y')}"
)
print(query)


## 3. Run one Tavily search

This cell uses one basic-search credit per run. `include_domains` limits results to the restaurant’s site. `include_raw_content` requests page content, which may be absent or incomplete. We explicitly disable automatic parameter selection and generated answers. No OpenAI request is made.

If Tavily finds no usable results, inspect that outcome before spending more credits or changing providers.

In [ ]:
if not tavily_key:
    raise ValueError("Add TAVILY_API_KEY to .env and rerun the setup cell")

response = httpx.post(
    "https://api.tavily.com/search",
    headers={"Authorization": f"Bearer {tavily_key}"},
    json={
        "query": query,
        "search_depth": "basic",
        "auto_parameters": False,
        "max_results": 3,
        "include_domains": [restaurant["domain"]],
        "include_raw_content": True,
        "include_answer": False,
        "include_usage": True,
    },
    timeout=60,
)
response.raise_for_status()
tavily_result = response.json()

print("Sources found:", len(tavily_result.get("results", [])))
print("Usage:", tavily_result.get("usage", "Not returned"))


## 4. Inspect the results without searching again

Look for the requested date or a week that contains it, meal names and prices. A relevant-looking title or high search score is not date evidence. This cell prints a preview; use the following inspection cell for the full content of a chosen result. Images or PDFs may still require a separate reader.

If the results are useful, the next experiment will send this source content to an extraction model without OpenAI web search. That extraction would still incur model-token charges.

In [ ]:
for index, source in enumerate(tavily_result.get("results", [])):
    content = source.get("raw_content") or source.get("content") or ""
    print("\nResult", index)
    print("Title:", source.get("title"))
    print("URL:", source.get("url"))
    print("Raw content available:", bool(source.get("raw_content")))
    print("Characters:", len(content))
    print(content[:4000])


In [ ]:
# Change 0 to inspect another result. This makes no API request.
sources = tavily_result.get("results", [])
if sources:
    source = sources[0]
    print(source.get("raw_content") or source.get("content") or "No content returned")
else:
    print("No sources returned for this query")
